> LLM 倾向于关注 Prompt 开头和结尾的信息，而常常忽略中间的信息（即 "Lost in the Middle" 现象）

### rerank 必要性

> Rerank 模型（重排序模型）通常基于 Cross-Encoder（交叉编码器） 架构。与向量检索（Embedding）分别处理 Query and Document 不同，Rerank 模型的特点是**“成对输入，打分输出”**。

- 弥补向量检索（Embedding）的“语义损失”
    - 在 RAG 的第一步（检索）中，为了速度，我们通常使用**双编码器（Bi-Encoder）**将文档压缩成向量。虽然速度快，但存在精度缺陷：
        - 信息压缩损失： 将一段几百字的文本压缩成一个固定长度的向量（如 768 维），必然会丢失很多细节。
        - 无法处理复杂语义： 向量相似度往往基于“模糊的语义接近”，容易被关键词误导。例如，“我不喜欢苹果”和“我喜欢苹果”在向量空间中可能靠得很近，但意思完全相反。
    - Rerank 的作用： Rerank 通常使用交叉编码器（Cross-Encoder）。它不预先压缩文档，而是将“用户的问题”和“候选文档”拼接在一起，让模型逐字逐句地对比分析。这种方式计算量大、速度慢，但能精准识别逻辑关系、否定词和细微差别，从而把真正相关的文档排到前面。
- 解决 LLM 的“长窗口迷失” (Lost in the Middle)，由于位置导致的注意力偏差
    - 检索回来的 Top 10 文档中，最正确的那条可能排在第 7 位。如果直接喂给 LLM，它被放在中间，很容易被模型“视而不见”，导致幻觉。
    - Rerank 会根据相关性分数重新排序，强制将相关性最高的文档（Gold chunks）放在 Prompt 的最前面（或最后面），确保 LLM 能够“看清”关键证据。
- RAG 系统需要在**召回率（Recall）和精准率（Precision）**之间做权衡。
    - 初排（Retrieval）： 目标是宁滥勿缺。为了不漏掉正确答案，我们通常会检索 Top 50 甚至 Top 100 个片段。
    - 生成（Generation）： 目标是精准。如果把这 100 个片段全给 LLM，不仅 Token 成本极高，推理速度会变慢，还会引入大量噪音（无关信息），干扰 LLM 回答。
    - Rerank 的作用： Rerank 充当了过滤器。它从 Top 100 中精选出 Top 3 或 Top 5。这样既享受了大规模检索的广度，又保证了输入给 LLM 的数据的纯度，实现了高召回 + 高精度 + 低成本。